## Goal
Pack 14 Christmas tree polygons into the smallest square. Target score: 0.3696

Score formula: `max(width, height)^2 / N`

In [ ]:
# =========================================================
# 0) Setup: clone sparrow + install rust nightly (one time)
# =========================================================
import os, shutil, subprocess, time, glob, json, math, heapq
from pathlib import Path

if not Path("sparrow").exists():
    !git clone https://github.com/SmartManoj/sparrow

# rustup install (idempotent-ish)
if not Path("/root/.cargo/bin/rustup").exists():
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

os.environ["PATH"] += ":/root/.cargo/bin"

# set nightly (required by sparrow for some features)
!rustup default nightly

print("✅ Setup done.")


In [ ]:
# =========================================================
# 1) Utilities
# =========================================================
SCALE = 1000.0

TREE_VERTS = [
    (0.0, 0.8), (0.125, 0.5), (0.0625, 0.5), (0.2, 0.25), (0.1, 0.25),
    (0.35, 0.0), (0.075, 0.0), (0.075, -0.2), (-0.075, -0.2), (-0.075, 0.0),
    (-0.35, 0.0), (-0.1, 0.25), (-0.2, 0.25), (-0.0625, 0.5), (-0.125, 0.5)
]

def transform_point(x, y, tx, ty, deg):
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    rx = x * c - y * s
    ry = x * s + y * c
    return rx + tx, ry + ty

def calc_score(placements):
    min_x = min_y = float('inf')
    max_x = max_y = float('-inf')
    for tx, ty, deg in placements:
        for vx, vy in TREE_VERTS:
            px, py = transform_point(vx, vy, tx, ty, deg)
            min_x = min(min_x, px)
            max_x = max(max_x, px)
            min_y = min(min_y, py)
            max_y = max(max_y, py)
    width = max_x - min_x
    height = max_y - min_y
    side = max(width, height)
    return side ** 2 / len(placements), side, width, height

def write_sparrow_input_json(N, strip_height_int, name, out_json_path):
    data = {
        "name": name,
        "items": [{
            "id": 0,
            "demand": int(N),
            "shape": {"type":"simple_polygon","data":[
                [0,800],[125,500],[62.5,500],[200,250],[100,250],[350,0],
                [75,0],[75,-200],[-75,-200],[-75,0],[-350,0],[-100,250],
                [-200,250],[-62.5,500],[-125,500]
            ]}
        }],
        "strip_height": int(strip_height_int)
    }
    with open(out_json_path, "w") as f:
        json.dump(data, f)

import subprocess
from pathlib import Path

def run_sparrow(input_json_path, t_sec=120, seed=42, verbose=False, log_lines=40):
    """
    Run sparrow and return final json path.
    If verbose=True, print last log_lines lines (stdout+stderr).
    """
    out_dir = Path("sparrow/output")
    out_dir.mkdir(parents=True, exist_ok=True)
    before = set(out_dir.glob("final_*.json"))

    cmd = (
        f"cd sparrow && "
        f"cargo run --release --features=simd,only_final_svg -- "
        f"-i ../{input_json_path} -t {int(t_sec)} -s {int(seed)}"
    )

    # ✅ 用 subprocess 跑 bash -lc
    if verbose:
        print("[CMD]", cmd)
        res = subprocess.run(
            ["bash", "-lc", cmd],
            capture_output=True,
            text=True
        )
        logs = (res.stdout or "") + (res.stderr or "")
        if log_lines is None:
            print(logs)
        else:
            lines = logs.splitlines()
            print("\n".join(lines[-int(log_lines):]))
    else:
        subprocess.run(
            ["bash", "-lc", cmd],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

    after = set(out_dir.glob("final_*.json"))
    new_files = list(after - before)

    if len(new_files) == 0:
        candidates = sorted(out_dir.glob("final_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
        if not candidates:
            return None
        return str(candidates[0])

    new_files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(new_files[0])



def load_sparrow_solution(final_json_path):
    with open(final_json_path, "r") as f:
        data = json.load(f)
    sol = data.get("solution", data)
    placements = []
    for item in sol["layout"]["placed_items"]:
        t = item["transformation"]
        x = t["translation"][0] / SCALE
        y = t["translation"][1] / SCALE
        rot = t["rotation"]
        placements.append((x, y, rot))
    return placements

def load_submission(csv_path):
    data = {}
    with open(csv_path, "r") as f:
        _ = f.readline()
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            id_ = parts[0]
            x = float(parts[1][1:])
            y = float(parts[2][1:])
            deg = float(parts[3][1:])
            data[id_] = (x, y, deg)
    return data

def extract_group(data, group):
    placements = []
    i = 0
    while f"{group}_{i}" in data:
        placements.append(data[f"{group}_{i}"])
        i += 1
    return placements

def save_submission(data, out_path):
    def sort_key(k):
        g, idx = k.split("_")
        return (int(g), int(idx))
    keys = sorted(data.keys(), key=sort_key)
    with open(out_path, "w") as f:
        f.write("id,x,y,deg\n")
        for k in keys:
            x, y, deg = data[k]
            f.write(f"{k},s{x:.17f},s{y:.17f},s{deg:.17f}\n")

def merge_solution_to_new_csv(base_submission_path, out_path, group, final_json_path, N_expected):
    """
    强制 merge：不管新解是否更好，都替换 group 并另存为 out_path
    """
    data = load_submission(base_submission_path)
    old = extract_group(data, group)
    if not old:
        raise RuntimeError(f"Group {group} not found in base submission!")

    new_placements = load_sparrow_solution(final_json_path)
    if len(new_placements) != N_expected:
        raise RuntimeError(f"Sparrow placements count mismatch: got={len(new_placements)} expected={N_expected}")

    # replace
    for i, (x, y, deg) in enumerate(new_placements):
        data[f"{group}_{i}"] = (x, y, deg)

    save_submission(data, out_path)
    return out_path


In [ ]:
# =========================================================
# 2) Generate Top-K initial points for ONE N, and export submission_k.csv
# =========================================================

# 你要跑的 N（只跑一个）
N = 34
group = f"{N:03d}"

# 输出 TopK 个版本
TOPK = 12   # <- 你要多少个 submission_k，就设多少

# 你的 baseline submission（改成你的真实输入）
BASE_SUB_PATH = "/kaggle/input/merging/submission.csv"

# 把 baseline 拷贝到当前目录，作为 merge 的 base
!cp "{BASE_SUB_PATH}" "submission_base.csv"
print("✅ Loaded baseline -> submission_base.csv")

# 你的 side_length 估计，用来算 H0
BEST_SIDE = {
    57: 4.484638629210352640,
    58: 4.496813336078086656,
    65: 4.781086052083545088,
    134: 6.759755975547483136,
    151: 7.111531351091566592,
    34: 3.6749758724227594
}

SEEDS = [42, 100, 200, 300, 400, 500, 777, 888, 999, 2026]
COARSE_DELTAS = [-20, -10, 0, 10, 20]

T_COARSE = 120
T_REFINE = 300

H0 = int(round(BEST_SIDE[N] * 1000))
heights = [H0 + d for d in COARSE_DELTAS]

print("="*80)
print(f"🎯 N={N} group={group}  TOPK={TOPK}")
print(f"H0={H0}, heights={heights}")
print("="*80)

# 用 min-heap 存 TopK（score 越小越好）
# heap item: (-score, info_dict)
topk_heap = []

def push_topk(score, info):
    # 用负号让 heap 按 score 最大先弹（我们维护 topK 最小）
    item = (-score, info)
    if len(topk_heap) < TOPK:
        heapq.heappush(topk_heap, item)
    else:
        # heap[0] 是当前 topK 里“最差的那个”（score 最大 -> -score 最小）
        if -topk_heap[0][0] > score:
            heapq.heapreplace(topk_heap, item)

# ---------- COARSE SWEEP ----------
for H in heights:
    for seed in SEEDS:
        name = f"n{N:03d}_h{H}_s{seed}"
        in_path = f"{name}.json"
        write_sparrow_input_json(N, H, name, in_path)

        final_json = run_sparrow(in_path, t_sec=T_COARSE, seed=seed, verbose=False)
        if final_json is None:
            print(f"⚠️ fail: H={H} seed={seed}")
            continue

        placements = load_sparrow_solution(final_json)
        score, side, w, h = calc_score(placements)

        push_topk(score, {
            "phase": "coarse",
            "H": H,
            "seed": seed,
            "score": score,
            "side": side,
            "w": w,
            "h": h,
            "final_json": final_json
        })

print("\n✅ coarse done, current heap size =", len(topk_heap))

# ---------- REFINE (可选)：对 coarse 中的 best few 进行更长时间 ----------
# 取 heap 里目前最好的 3 个 height 再 refine
tmp = sorted([(-s, info) for (s, info) in topk_heap], key=lambda x: x[0])  # score升序
best_refine = tmp[:3]

for score0, info0 in best_refine:
    bestH = info0["H"]
    for seed in SEEDS[:5]:
        name = f"n{N:03d}_REFINE_h{bestH}_s{seed}"
        in_path = f"{name}.json"
        write_sparrow_input_json(N, bestH, name, in_path)

        final_json = run_sparrow(in_path, t_sec=T_REFINE, seed=seed, verbose=False)
        if final_json is None:
            continue

        placements = load_sparrow_solution(final_json)
        score, side, w, h = calc_score(placements)

        push_topk(score, {
            "phase": "refine",
            "H": bestH,
            "seed": seed,
            "score": score,
            "side": side,
            "w": w,
            "h": h,
            "final_json": final_json
        })

print("\n✅ refine done, heap size =", len(topk_heap))

# ---------- EXPORT submission_k.csv ----------
topk_sorted = sorted([(-s, info) for (s, info) in topk_heap], key=lambda x: x[0])  # score升序
print("\n" + "="*80)
print("✅ TOPK RESULTS (Sparrow only)")
print("="*80)
for rank, (score, info) in enumerate(topk_sorted, 1):
    print(f"[{rank}] score={score:.12f} phase={info['phase']} H={info['H']} seed={info['seed']} side={info['side']:.6f}")

# 强制 merge，每个 rank 输出一个 submission_rank.csv
os.makedirs("topk_outputs", exist_ok=True)

meta = {
    "N": N,
    "group": group,
    "TOPK": TOPK,
    "items": []
}

for rank, (score, info) in enumerate(topk_sorted, 1):
    out_csv = f"topk_outputs/submission_{rank}.csv"
    merge_solution_to_new_csv(
        base_submission_path="submission_base.csv",
        out_path=out_csv,
        group=group,
        final_json_path=info["final_json"],
        N_expected=N
    )
    meta["items"].append({
        "rank": rank,
        "score": score,
        "phase": info["phase"],
        "H": info["H"],
        "seed": info["seed"],
        "final_json": info["final_json"],
        "out_csv": out_csv
    })
    print(f"✅ wrote: {out_csv}")

with open("topk_outputs/topk_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\n✅ DONE. outputs in: topk_outputs/")


In [ ]:
# =========================================================
# 2) Generate Top-K initial points for ONE N, and export submission_k.csv
# =========================================================

# 你要跑的 N（只跑一个）
N = 29
group = f"{N:03d}"

# 输出 TopK 个版本
TOPK = 8   # <- 你要多少个 submission_k，就设多少

# 你的 baseline submission（改成你的真实输入）
BASE_SUB_PATH = "/kaggle/input/intergration-of-existing-result-current-best/submission.csv"

# 把 baseline 拷贝到当前目录，作为 merge 的 base
!cp "{BASE_SUB_PATH}" "submission_base.csv"
print("✅ Loaded baseline -> submission_base.csv")

# 你的 side_length 估计，用来算 H0
BEST_SIDE = {
    57: 4.484638629210352640,
    58: 4.496813336078086656,
    65: 4.781086052083545088,
    134: 6.759755975547483136,
    151: 7.111531351091566592,
    17: 2.508124,
    29: 3.25
}

SEEDS = [42, 100, 200, 300, 400, 500, 777, 888, 999, 2026]
COARSE_DELTAS = [-20, -10, 0, 10, 20]

T_COARSE = 120
T_REFINE = 300

H0 = int(round(BEST_SIDE[N] * 1000))
heights = [H0 + d for d in COARSE_DELTAS]

print("="*80)
print(f"🎯 N={N} group={group}  TOPK={TOPK}")
print(f"H0={H0}, heights={heights}")
print("="*80)

# 用 min-heap 存 TopK（score 越小越好）
# heap item: (-score, info_dict)
topk_heap = []

def push_topk(score, info):
    # 用负号让 heap 按 score 最大先弹（我们维护 topK 最小）
    item = (-score, info)
    if len(topk_heap) < TOPK:
        heapq.heappush(topk_heap, item)
    else:
        # heap[0] 是当前 topK 里“最差的那个”（score 最大 -> -score 最小）
        if -topk_heap[0][0] > score:
            heapq.heapreplace(topk_heap, item)

# ---------- COARSE SWEEP ----------
for H in heights:
    for seed in SEEDS:
        name = f"n{N:03d}_h{H}_s{seed}"
        in_path = f"{name}.json"
        write_sparrow_input_json(N, H, name, in_path)

        final_json = run_sparrow(in_path, t_sec=T_COARSE, seed=seed, verbose=False)
        if final_json is None:
            print(f"⚠️ fail: H={H} seed={seed}")
            continue

        placements = load_sparrow_solution(final_json)
        score, side, w, h = calc_score(placements)

        push_topk(score, {
            "phase": "coarse",
            "H": H,
            "seed": seed,
            "score": score,
            "side": side,
            "w": w,
            "h": h,
            "final_json": final_json
        })

print("\n✅ coarse done, current heap size =", len(topk_heap))

# ---------- REFINE (可选)：对 coarse 中的 best few 进行更长时间 ----------
# 取 heap 里目前最好的 3 个 height 再 refine
tmp = sorted([(-s, info) for (s, info) in topk_heap], key=lambda x: x[0])  # score升序
best_refine = tmp[:3]

for score0, info0 in best_refine:
    bestH = info0["H"]
    for seed in SEEDS[:5]:
        name = f"n{N:03d}_REFINE_h{bestH}_s{seed}"
        in_path = f"{name}.json"
        write_sparrow_input_json(N, bestH, name, in_path)

        final_json = run_sparrow(in_path, t_sec=T_REFINE, seed=seed, verbose=False)
        if final_json is None:
            continue

        placements = load_sparrow_solution(final_json)
        score, side, w, h = calc_score(placements)

        push_topk(score, {
            "phase": "refine",
            "H": bestH,
            "seed": seed,
            "score": score,
            "side": side,
            "w": w,
            "h": h,
            "final_json": final_json
        })

print("\n✅ refine done, heap size =", len(topk_heap))

# ---------- EXPORT submission_k.csv ----------
topk_sorted = sorted([(-s, info) for (s, info) in topk_heap], key=lambda x: x[0])  # score升序
print("\n" + "="*80)
print("✅ TOPK RESULTS (Sparrow only)")
print("="*80)
for rank, (score, info) in enumerate(topk_sorted, 1):
    print(f"[{rank}] score={score:.12f} phase={info['phase']} H={info['H']} seed={info['seed']} side={info['side']:.6f}")

# 强制 merge，每个 rank 输出一个 submission_rank.csv
os.makedirs("topk_outputs", exist_ok=True)

meta = {
    "N": N,
    "group": group,
    "TOPK": TOPK,
    "items": []
}

for rank, (score, info) in enumerate(topk_sorted, 1):
    out_csv = f"topk_outputs/submission_{rank}.csv"
    merge_solution_to_new_csv(
        base_submission_path="submission_base.csv",
        out_path=out_csv,
        group=group,
        final_json_path=info["final_json"],
        N_expected=N
    )
    meta["items"].append({
        "rank": rank,
        "score": score,
        "phase": info["phase"],
        "H": info["H"],
        "seed": info["seed"],
        "final_json": info["final_json"],
        "out_csv": out_csv
    })
    print(f"✅ wrote: {out_csv}")

with open("topk_outputs/topk_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\n✅ DONE. outputs in: topk_outputs/")
